# Evaluation 2 — Random Forest Development

This notebook is **Sanugi's primary algorithm contribution**. It justifies Random Forest, establishes the common baseline, evaluates the approved General/City/Resort training scopes, performs controlled randomized tuning, compares baseline and tuned grouped-CV evidence, generates training-only grouped out-of-fold (OOF) predictions, investigates the City/Resort novelty, and freezes one configuration per scope for later evaluation.

**NO OUTER-TEST MODEL PERFORMANCE IS CALCULATED IN THIS NOTEBOOK.** Outer-test positions are reconstructed only to verify the approved split size; they are never used for fitting, prediction, metrics, or model decisions.

## 1. Imports and shared infrastructure

The existing project modules remain authoritative. Predictor groups are partition helpers—not model features, booking IDs, or customer IDs. All learned preprocessing remains inside each fold-fitted pipeline.

In [ ]:
from pathlib import Path
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, cross_validate

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(root for root in root_candidates if (root / 'src' / 'modeling.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import TARGET, prepare_training_table, split_features_target, create_predictor_groups
from src.modeling import (
    RANDOM_STATE, PRIMARY_SCORING, COMMON_SCORING, MODEL_SCOPES, make_inner_cv,
    create_training_scopes, summarize_training_scopes, build_model_pipeline, CV_RESULT_COLUMNS,
)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
CV_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'random_forest_cv_results.csv'
OOF_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'random_forest_oof_results.csv'
RUN_STARTED = time.perf_counter()
captured_warning_messages = []
assert PRIMARY_SCORING == 'f1'
print(f'Project root: {PROJECT_ROOT}')
print(f'Primary scoring metric: {PRIMARY_SCORING}')

## 2. Reconstruct the approved outer-training population

The approved five-fold `StratifiedGroupKFold` is recreated with shuffling, seed 42, and the first split only. Only outer-training objects are created for development; test positions exist solely for row-count and group-separation verification.

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
training_table = prepare_training_table(raw_df)
X, y = split_features_target(training_table)
predictor_groups = create_predictor_groups(X)

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_positions, test_positions = next(outer_splitter.split(X, y, groups=predictor_groups))
X_train_outer = X.iloc[train_positions].copy()
y_train_outer = y.iloc[train_positions].copy()
groups_train_outer = predictor_groups.iloc[train_positions].copy()

assert len(X) == 119210
assert len(train_positions) == len(X_train_outer) == 95375
assert len(test_positions) == 23835
assert X_train_outer.shape[1] == 30
assert TARGET not in X_train_outer.columns
assert 'predictor_group' not in X_train_outer.columns
assert not set(groups_train_outer.unique()).intersection(predictor_groups.iloc[test_positions].unique())
display(pd.DataFrame({'measure': ['Modelling rows', 'Outer-training rows', 'Outer-test positions', 'Unencoded predictors'], 'value': [len(X), len(X_train_outer), len(test_positions), X_train_outer.shape[1]]}))

## 3. Approved training scopes

City and Resort are subsets of the same fixed outer-training population. Every scope retains the shared 30-column schema, including `hotel` in each specialist scope.

In [ ]:
scopes = create_training_scopes(X_train_outer, y_train_outer, groups_train_outer)
scope_summary = summarize_training_scopes(scopes)
assert tuple(scopes) == MODEL_SCOPES
assert scope_summary.set_index('scope')['rows'].to_dict() == {'General': 95375, 'City': 63044, 'Resort': 32331}
for scope_name in MODEL_SCOPES:
    assert scopes[scope_name]['X'].shape[1] == 30
    assert 'hotel' in scopes[scope_name]['X'].columns
    assert 'predictor_group' not in scopes[scope_name]['X'].columns
display(scope_summary)

## 4. Random Forest rationale and baseline

Random Forest is suitable for binary classification because it combines many decision trees, captures nonlinear relationships and interactions among booking characteristics, and averages/votes across trees to generally reduce variance compared with one tree. It is a robust ensemble baseline for complex tabular relationships, estimates cancellation probabilities, and provides an ensemble comparison against linear and single-tree approaches.

Its limitations include lower direct interpretability, greater runtime and memory use for large/deep forests, and sensitivity of cancellation detection to class imbalance. Ensemble size, tree complexity, feature sampling, and class weighting therefore matter. No claim is made that Random Forest will be the best algorithm. Feature importance may be useful later for interpretation, but it is not used for feature selection here.

Random Forest does not require numerical scaling in the same way some linear or distance-based methods do. Nevertheless, this project deliberately retains the approved common preprocessing pipeline for methodological consistency and leakage-safe fold-specific preprocessing.

The fixed baseline uses 200 trees for a reasonably stable ensemble, Gini splits, unrestricted depth, leaf size 1, square-root feature sampling, standard bootstrap bagging, no class weighting, and seed 42. Estimator-level `n_jobs=1` prevents nested oversubscription while outer CV/search uses `n_jobs=-1`. The default decision rule is used; no threshold tuning occurs.

## Baseline Random Forest Grouped-CV Results

Each result comes from a complete `preprocessor → classifier` pipeline. Imputation, scaling, categorical missing handling, one-hot vocabulary, and forest trees are learned independently inside every grouped training fold.

In [ ]:
BASELINE_PARAMETERS = {
    'n_estimators': 200, 'criterion': 'gini', 'max_depth': None,
    'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt',
    'bootstrap': True, 'class_weight': None, 'random_state': RANDOM_STATE, 'n_jobs': 1,
}
BASELINE_JSON = json.dumps(BASELINE_PARAMETERS, sort_keys=True)
METRICS = ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
cv_records = []

def preserve_warnings(caught, context):
    for item in caught:
        captured_warning_messages.append(f'{context} | {item.category.__name__}: {item.message}')

def make_cv_record(scope_name, stage, summary, *, tuning_method=None, best_parameters=None, observation=''):
    scope = scopes[scope_name]
    record = {column: None for column in CV_RESULT_COLUMNS}
    record.update({
        'algorithm': 'Random Forest', 'scope': scope_name, 'stage': stage,
        'cv_strategy': 'StratifiedGroupKFold(5)', 'random_state': RANDOM_STATE,
        'parameters': BASELINE_JSON if stage == 'baseline' else json.dumps(summary['parameters'], sort_keys=True),
        'training_rows': len(scope['X']), 'predictor_groups': scope['groups'].nunique(),
        'tuning_method': tuning_method, 'best_parameters': best_parameters, 'observation': observation,
    })
    for metric in METRICS:
        record[f'mean_cv_{metric}'] = float(summary[f'mean_{metric}'])
        record[f'std_cv_{metric}'] = float(summary[f'std_{metric}'])
    return record

baseline_summaries = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = RandomForestClassifier(**BASELINE_PARAMETERS)
    assert estimator.n_jobs == 1
    pipeline = build_model_pipeline(scope['X'], estimator)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        scores = cross_validate(
            pipeline, scope['X'], scope['y'], groups=scope['groups'], cv=make_inner_cv(),
            scoring=COMMON_SCORING, n_jobs=-1, return_train_score=False,
        )
    preserve_warnings(caught, f'baseline {scope_name}')
    summary = {}
    for metric in METRICS:
        summary[f'mean_{metric}'] = scores[f'test_{metric}'].mean()
        summary[f'std_{metric}'] = scores[f'test_{metric}'].std()
    baseline_summaries[scope_name] = summary
    cv_records.append(make_cv_record(scope_name, 'baseline', summary, observation='Fixed 200-tree baseline evaluated with grouped inner CV.'))

baseline_table = pd.DataFrame([{'scope': s, **baseline_summaries[s]} for s in MODEL_SCOPES])
display(baseline_table)

## 5. Controlled randomized tuning

The revised compute-safe search space has 2 ensemble sizes × 3 depth options × 3 leaf sizes × 2 feature-sampling options × 3 class-weight strategies = **108 combinations**. `RandomizedSearchCV` samples exactly **6 reproducible candidates per scope** with seed 42.

This reduction is deliberate: the original 12-candidate space included 400-tree forests and proved unnecessarily expensive on the available machine. No model results from that incomplete run were used to choose these settings. The revised design still tunes ensemble size, tree depth, terminal-leaf size, feature sampling, and class weighting under the same grouped 5-fold validation protocol.

`n_estimators` controls ensemble size and runtime; `max_depth` controls individual-tree complexity; `min_samples_leaf` limits highly specific terminal rules; `max_features` affects tree diversity; and `class_weight` tests imbalance handling. `balanced_subsample` recomputes weights within each bootstrap sample. Criterion, split size, bootstrap behavior, preprocessing, feature set, and threshold remain fixed.


In [ ]:
PARAM_DISTRIBUTIONS = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 12, 24],
    'classifier__min_samples_leaf': [1, 5, 10],
    'classifier__max_features': ['sqrt', 0.5],
    'classifier__class_weight': [None, 'balanced', 'balanced_subsample'],
}
THEORETICAL_COMBINATIONS = int(np.prod([len(values) for values in PARAM_DISTRIBUTIONS.values()]))
assert THEORETICAL_COMBINATIONS == 108
best_parameters_by_scope = {}
tuned_summaries = {}
candidate_counts = {}

for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = RandomForestClassifier(**BASELINE_PARAMETERS)
    assert estimator.n_jobs == 1
    pipeline = build_model_pipeline(scope['X'], estimator)
    search = RandomizedSearchCV(
        estimator=pipeline, param_distributions=PARAM_DISTRIBUTIONS, n_iter=6,
        scoring=COMMON_SCORING, refit=PRIMARY_SCORING, cv=make_inner_cv(),
        random_state=RANDOM_STATE, n_jobs=-1, return_train_score=False,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        search.fit(scope['X'], scope['y'], groups=scope['groups'])
    preserve_warnings(caught, f'tuning {scope_name}')
    best_index = search.best_index_
    candidate_counts[scope_name] = len(search.cv_results_['params'])
    assert candidate_counts[scope_name] == 6
    assert search.best_estimator_.named_steps['classifier'].n_jobs == 1
    best_parameters_by_scope[scope_name] = dict(search.best_params_)
    summary = {'parameters': {
        'n_estimators': search.best_params_['classifier__n_estimators'],
        'criterion': 'gini',
        'max_depth': search.best_params_['classifier__max_depth'],
        'min_samples_split': 2,
        'min_samples_leaf': search.best_params_['classifier__min_samples_leaf'],
        'max_features': search.best_params_['classifier__max_features'],
        'bootstrap': True,
        'class_weight': search.best_params_['classifier__class_weight'],
        'random_state': RANDOM_STATE, 'n_jobs': 1,
    }}
    for metric in METRICS:
        summary[f'mean_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_index]
        summary[f'std_{metric}'] = search.cv_results_[f'std_test_{metric}'][best_index]
    assert np.isclose(summary['mean_f1'], search.best_score_)
    tuned_summaries[scope_name] = summary
    cv_records.append(make_cv_record(
        scope_name, 'tuned', summary, tuning_method='RandomizedSearchCV',
        best_parameters=json.dumps(search.best_params_, sort_keys=True),
        observation='Best F1 configuration among 6 reproducibly sampled candidates.',
    ))
    del search

tuned_table = pd.DataFrame([
    {'scope': s, 'candidates': candidate_counts[s], 'best_parameters': json.dumps(best_parameters_by_scope[s], sort_keys=True),
     **{k: v for k, v in tuned_summaries[s].items() if k != 'parameters'}}
    for s in MODEL_SCOPES
])
display(tuned_table)


## 6. Baseline versus tuned validation performance

Changes are descriptive and interpreted within each scope. Different hotel populations are not treated as direct model-quality comparisons, and no statistical significance is claimed. Negative changes, if present, are retained and reported.

In [ ]:
comparison_rows = []
for scope_name in MODEL_SCOPES:
    baseline = baseline_summaries[scope_name]
    tuned = tuned_summaries[scope_name]
    comparison_rows.append({
        'scope': scope_name,
        'baseline_f1': baseline['mean_f1'], 'tuned_f1': tuned['mean_f1'], 'f1_change': tuned['mean_f1'] - baseline['mean_f1'],
        'baseline_recall': baseline['mean_recall'], 'tuned_recall': tuned['mean_recall'], 'recall_change': tuned['mean_recall'] - baseline['mean_recall'],
        'baseline_roc_auc': baseline['mean_roc_auc'], 'tuned_roc_auc': tuned['mean_roc_auc'], 'roc_auc_change': tuned['mean_roc_auc'] - baseline['mean_roc_auc'],
    })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)

cv_results_df = pd.DataFrame(cv_records, columns=CV_RESULT_COLUMNS)
assert len(cv_results_df) == 6
assert tuple(cv_results_df.columns) == CV_RESULT_COLUMNS
assert set(cv_results_df['stage']) == {'baseline', 'tuned'}
assert set(cv_results_df['scope']) == set(MODEL_SCOPES)
CV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cv_results_df.to_csv(CV_OUTPUT_PATH, index=False)
print(f'Saved {len(cv_results_df)} standardized CV records to {CV_OUTPUT_PATH}')

## 7. Frozen configurations and grouped OOF predictions

Only each search's selected parameter dictionary is retained. Every OOF fold creates and clones a fresh full pipeline, fits only fold-training rows, and predicts only fold-validation rows. Probabilities—not hard predictions—are used for ROC-AUC. Assertions enforce zero group overlap and exactly one validation prediction per row.

In [ ]:
def grouped_oof_predictions(X_scope, y_scope, groups_scope, selected_parameters, scope_name):
    hard_predictions = np.full(len(X_scope), -1, dtype=int)
    cancellation_probabilities = np.full(len(X_scope), np.nan, dtype=float)
    validation_counts = np.zeros(len(X_scope), dtype=int)
    fold_group_overlaps = []

    for fold_number, (train_idx, valid_idx) in enumerate(
        make_inner_cv().split(X_scope, y_scope, groups=groups_scope), start=1
    ):
        overlap = set(groups_scope.iloc[train_idx]).intersection(groups_scope.iloc[valid_idx])
        fold_group_overlaps.append(len(overlap))
        assert not overlap, f'{scope_name} fold {fold_number} has group overlap'
        estimator = RandomForestClassifier(
            n_estimators=selected_parameters['classifier__n_estimators'], criterion='gini',
            max_depth=selected_parameters['classifier__max_depth'], min_samples_split=2,
            min_samples_leaf=selected_parameters['classifier__min_samples_leaf'],
            max_features=selected_parameters['classifier__max_features'], bootstrap=True,
            class_weight=selected_parameters['classifier__class_weight'],
            random_state=RANDOM_STATE, n_jobs=1,
        )
        assert estimator.n_jobs == 1
        model = clone(build_model_pipeline(X_scope, estimator))
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            model.fit(X_scope.iloc[train_idx], y_scope.iloc[train_idx])
        preserve_warnings(caught, f'OOF {scope_name} fold {fold_number}')
        hard_predictions[valid_idx] = model.predict(X_scope.iloc[valid_idx])
        cancellation_probabilities[valid_idx] = model.predict_proba(X_scope.iloc[valid_idx])[:, 1]
        validation_counts[valid_idx] += 1

    assert np.all(validation_counts == 1)
    assert np.all(hard_predictions >= 0)
    assert np.isfinite(cancellation_probabilities).all()
    assert max(fold_group_overlaps) == 0
    return hard_predictions, cancellation_probabilities, validation_counts, fold_group_overlaps

oof_predictions = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    oof_predictions[scope_name] = grouped_oof_predictions(
        scope['X'], scope['y'], scope['groups'], best_parameters_by_scope[scope_name], scope_name,
    )
    print(f'{scope_name}: {len(scope["X"]):,} rows predicted exactly once; fold group overlaps = {oof_predictions[scope_name][3]}')

In [ ]:
def make_oof_metric_record(scope_name, evaluation_population, y_true, hard_predictions, probabilities):
    return {
        'algorithm': 'Random Forest', 'scope': scope_name, 'evaluation_population': evaluation_population,
        'stage': 'tuned_oof', 'rows': len(y_true),
        'accuracy': accuracy_score(y_true, hard_predictions),
        'precision': precision_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'recall': recall_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'f1': f1_score(y_true, hard_predictions, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probabilities),
        'parameters': json.dumps(best_parameters_by_scope[scope_name], sort_keys=True),
    }

general = scopes['General']
general_hard, general_probabilities = oof_predictions['General'][:2]
city_mask = general['X']['hotel'].eq('City Hotel').to_numpy()
resort_mask = general['X']['hotel'].eq('Resort Hotel').to_numpy()
oof_records = [
    make_oof_metric_record('General', 'Overall', general['y'], general_hard, general_probabilities),
    make_oof_metric_record('General', 'City', general['y'].iloc[np.flatnonzero(city_mask)], general_hard[city_mask], general_probabilities[city_mask]),
    make_oof_metric_record('General', 'Resort', general['y'].iloc[np.flatnonzero(resort_mask)], general_hard[resort_mask], general_probabilities[resort_mask]),
    make_oof_metric_record('City', 'City', scopes['City']['y'], oof_predictions['City'][0], oof_predictions['City'][1]),
    make_oof_metric_record('Resort', 'Resort', scopes['Resort']['y'], oof_predictions['Resort'][0], oof_predictions['Resort'][1]),
]
oof_results_df = pd.DataFrame(oof_records)
expected_pairs = {('General', 'Overall'), ('General', 'City'), ('General', 'Resort'), ('City', 'City'), ('Resort', 'Resort')}
assert len(oof_results_df) == 5
assert set(zip(oof_results_df['scope'], oof_results_df['evaluation_population'])) == expected_pairs
assert set(oof_results_df['stage']) == {'tuned_oof'}
oof_results_df.to_csv(OOF_OUTPUT_PATH, index=False)
display(oof_results_df)
print(f'Saved {len(oof_results_df)} OOF records to {OOF_OUTPUT_PATH}')

## 8. Training-only novelty comparison

Specialist models are compared only with the General model on the same hotel population. Positive differences mean a higher specialist OOF metric. These are observed grouped training-only validation differences, not proof of final superiority.

In [ ]:
novelty_rows = []
for hotel_scope in ('City', 'Resort'):
    general_row = oof_results_df.query("scope == 'General' and evaluation_population == @hotel_scope").iloc[0]
    specialist_row = oof_results_df.query('scope == @hotel_scope and evaluation_population == @hotel_scope').iloc[0]
    novelty_rows.append({
        'population': hotel_scope,
        'general_f1': general_row['f1'], 'specialist_f1': specialist_row['f1'], 'specialist_minus_general_f1': specialist_row['f1'] - general_row['f1'],
        'general_recall': general_row['recall'], 'specialist_recall': specialist_row['recall'], 'specialist_minus_general_recall': specialist_row['recall'] - general_row['recall'],
        'general_roc_auc': general_row['roc_auc'], 'specialist_roc_auc': specialist_row['roc_auc'], 'specialist_minus_general_roc_auc': specialist_row['roc_auc'] - general_row['roc_auc'],
    })
novelty_comparison = pd.DataFrame(novelty_rows)
display(novelty_comparison)

## Random Forest Development Findings

In [ ]:
def forest_interpretation(scope_name):
    params = best_parameters_by_scope[scope_name]
    trees = params['classifier__n_estimators']
    depth = params['classifier__max_depth']
    leaf = params['classifier__min_samples_leaf']
    features = params['classifier__max_features']
    weight = params['classifier__class_weight']
    trees_text = 'the 200-tree ensemble size' if trees == 200 else 'a smaller 100-tree ensemble'
    depth_text = f'an explicit maximum depth of {depth}' if depth is not None else 'the unrestricted-depth option from the sampled candidates'
    leaf_text = f'larger terminal leaves of at least {leaf} records, reducing highly specific splits' if leaf > 1 else 'minimum leaf size 1'
    weight_text = (
        'no class reweighting' if weight is None else
        ('bootstrap-sample-specific balanced weighting' if weight == 'balanced_subsample' else 'balanced class weighting')
    )
    return f"{scope_name}: selected {trees_text}, {depth_text}, {leaf_text}, max_features={features!r}, and {weight_text}."

finding_lines = []
for scope_name in MODEL_SCOPES:
    f1_change = tuned_summaries[scope_name]['mean_f1'] - baseline_summaries[scope_name]['mean_f1']
    effect = 'improved' if f1_change > 0 else ('reduced' if f1_change < 0 else 'did not change')
    finding_lines.append(
        f"{scope_name} baseline F1={baseline_summaries[scope_name]['mean_f1']:.4f}; tuned F1={tuned_summaries[scope_name]['mean_f1']:.4f}; "
        f"the sampled search {effect} F1 by {f1_change:+.4f}. " + forest_interpretation(scope_name)
    )
for row in novelty_rows:
    finding_lines.append(
        f"{row['population']} same-population OOF specialist-minus-General differences: "
        f"F1={row['specialist_minus_general_f1']:+.4f}, Recall={row['specialist_minus_general_recall']:+.4f}, "
        f"ROC-AUC={row['specialist_minus_general_roc_auc']:+.4f}."
    )
finding_lines.extend([
    'Random Forest provides nonlinear, interaction-aware probability estimates with variance reduction through ensembling, but is less directly interpretable and more computationally expensive than a small single model.',
    'Feature importance is deferred as a possible later interpretation tool and is not used for feature selection or modelling decisions here.',
    'No other algorithm, threshold adjustment, or outer-test result is used in this notebook.',
    'The Random Forest configurations are now frozen for later final outer-holdout evaluation and cross-algorithm comparison.',
])
display(Markdown('\n'.join(f'- {line}' for line in finding_lines)))

In [ ]:
elapsed_minutes = (time.perf_counter() - RUN_STARTED) / 60
print(f'Notebook modelling runtime: {elapsed_minutes:.2f} minutes')
if captured_warning_messages:
    print('Captured Random Forest warnings require review:')
    for message in captured_warning_messages:
        print('-', message)
else:
    print('No Random Forest warnings occurred.')
print('Verification complete: 119,210 modelling rows; 95,375 outer-training rows; 23,835 outer-test positions.')
print('Theoretical search space: 108; each scope evaluated exactly 6 sampled candidates.')
print('All OOF rows were predicted exactly once and every OOF fold had zero group overlap.')
print('No outer-test model prediction or metric was calculated; no threshold tuning was performed.')